# Pasqal Cloud: first-job environment test

Follow [Pasqal's first-job guide](https://docs.pasqal.com/cloud/first-job/) using the project's installed packages. This builds a four-qubit Pulser sequence and submits two jobs (20 and 50 shots) to `RemoteEmuFreeBackend`. It does **not** submit to a QPU.

Run `uv sync --locked` in the repository terminal, select `.venv/bin/python` as the notebook kernel, and run the cells in order. Start Jupyter from the repository root (or a subdirectory). The first three code cells are offline checks; the connection and submission cells require network access and a Pasqal account with project access.

Credentials are loaded from the repository's ignored `.env`: `PASQAL_PROJECT_ID` and `PASQAL_USER_EMAIL` are required. Optionally set `PASQAL_PASSWORD`; otherwise enter your password in the hidden prompt. Existing process environment variables take precedence. Never paste credentials into cells or share credential-bearing errors or outputs.

In [1]:
import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from uuid import UUID

from dotenv import load_dotenv
from pasqal_cloud import PasqalCloudConnection, RemoteEmuFreeBackend
from pulser import Pulse, Register, Sequence
from pulser.devices import DigitalAnalogDevice

for package in ('pulser', 'pasqal-cloud', 'python-dotenv'):
    print(f'{package}: {version(package)}')


pulser: 1.9.1
pasqal-cloud: 0.24.0
python-dotenv: 1.2.4


In [2]:
load_dotenv()

required = ('PASQAL_PROJECT_ID', 'PASQAL_EMAIL')
missing = [name for name in required if not os.environ.get(name, '').strip()]
if missing:
    raise ValueError('Missing required environment variables: ' + ', '.join(missing))

project_id = os.environ['PASQAL_PROJECT_ID'].strip()
username = os.environ['PASQAL_EMAIL'].strip()
try:
    UUID(project_id)
except ValueError:
    raise ValueError('PASQAL_PROJECT_ID must be a project UUID from the Pasqal portal.') from None
if '@' not in username or username.startswith('<'):
    raise ValueError('PASQAL_EMAIL must contain your Pasqal account email.')
print('Required connection settings loaded; values are hidden.')


Required connection settings loaded; values are hidden.


## Build and validate the guide's sequence locally

Four atoms on a square, 5 micrometers apart, receive a 100 ns global Rydberg pulse. Its amplitude is the variable `omega_max`; detuning is 2 rad/us. Building both parameter sets checks local compatibility without contacting the cloud.

In [3]:
register = Register.square(2, spacing=5, prefix='q')
sequence = Sequence(register, DigitalAnalogDevice)
sequence.declare_channel('rydberg', 'rydberg_global')
omega_max = sequence.declare_variable('omega_max')
generic_pulse = Pulse.ConstantPulse(100, omega_max, 2, 0.0)
sequence.add(generic_pulse, 'rydberg')

job_params = [
    {'runs': 20, 'variables': {'omega_max': 6}},
    {'runs': 50, 'variables': {'omega_max': 10.5}},
]
for job in job_params:
    built_sequence = sequence.build(**job['variables'])
    assert not built_sequence.is_parametrized()
assert len(register.qubits) == 4
print('Local sequence checks passed: 4 qubits, 2 parameter sets, 70 total shots.')


Local sequence checks passed: 4 qubits, 2 parameter sets, 70 total shots.


## Connect to Pasqal Cloud

This uses the guide's default **global** region. Account authentication may happen lazily at submission; constructing the connection alone does not prove cloud access. A missing password triggers a hidden interactive prompt. Authentication/network errors are allowed to surface rather than being treated as successful tests.

In [4]:
password = os.environ.get('PASQAL_PASSWORD') or getpass('Pasqal account password: ')
if not password:
    raise ValueError('A Pasqal account password is required to connect.')
try:
    connection = PasqalCloudConnection(
        username=username,
        password=password,
        project_id=project_id,
    )
finally:
    del password
print('Connection initialized; submit the next cell to test cloud execution.')


Connection initialized; submit the next cell to test cloud execution.


## Submit the two jobs to the free cloud emulator

**Running this cell submits a new batch each time.** It uses your project's cloud access and any applicable quotas. `wait=True` waits for completion, which can take time in the queue. Interrupting the kernel does not necessarily cancel remote jobs. Check the [Pasqal portal](https://portal.pasqal.cloud/) before resubmitting; use the portal to monitor or cancel jobs.

In [5]:
backend = RemoteEmuFreeBackend(sequence=sequence, connection=connection)
results = backend.run(job_params=job_params, wait=True)
print('Cloud run returned. Inspect and validate the results below.')


Cloud run returned. Inspect and validate the results below.


## Inspect measured bitstrings and verify shot counts

Each bitstring has four bits, one per atom. Counts should total 20 and 50 for the two jobs, respectively. Quantum outcomes vary; this checks the output shape and shot totals, not particular bitstring probabilities. Do not rely on the result mapping's iteration order to identify jobs.

In [6]:
available_results = results.get_available_results()
assert len(available_results) == len(job_params), 'Expected a result for each submitted job.'
observed_shots = []
for job_id, result in available_results.items():
    counts = result.final_bitstrings
    assert counts, f'No measured bitstrings for job {job_id}.'
    assert all(len(bits) == 4 and set(bits) <= {'0', '1'} for bits in counts), 'Invalid bitstring shape.'
    assert all(isinstance(count, int) and count >= 0 for count in counts.values()), 'Invalid counts.'
    shots = sum(counts.values())
    observed_shots.append(shots)
    print(f'job-id: {job_id}; shots: {shots}')
    print(dict(sorted(counts.items())))

expected_shots = Counter(job['runs'] for job in job_params)
assert Counter(observed_shots) == expected_shots, 'Cloud shot totals do not match the submitted jobs.'
print('PASS: two cloud jobs returned valid four-bit measurements with 20 and 50 shots.')


job-id: 615efd6e-7ff0-416f-9269-30ac27993826; shots: 20
{'0000': 15, '0001': 2, '0010': 2, '0100': 1}
job-id: ff5a9138-24f2-424e-a781-b7c1734e07d4; shots: 50
{'0000': 18, '0001': 9, '0010': 4, '0100': 9, '0110': 2, '1000': 7, '1001': 1}
PASS: two cloud jobs returned valid four-bit measurements with 20 and 50 shots.
